# Job 2. 농장별 HPAI 위험도 예측 + Service Output + SHAP Top 3 + Batch 컬럼

이 노트북은 기존 Job 2 파일을 서비스 출력 계약에 맞게 수정한 버전입니다.

## 핵심 변경점

1. Job 2 시작 전에 `global_temp.job2_predictions_dryrun`을 읽지 않습니다.
2. Job 1에서 검증한 Signed LightGBM sklearn Pipeline을 그대로 사용합니다.
3. 모델 입력은 Signature 기준 30개 Feature만 사용합니다.
4. 예측은 Spark UDF로 전체 배치 처리합니다.
5. SHAP은 저장된 Pipeline 내부 `preprocessor`로 변환한 뒤 LightGBM estimator 기준으로 계산합니다.
6. 농장별 `riskFactors`에는 SHAP 상위 3개만 JSON 문자열로 저장합니다.
7. 입력 테이블은 앞 단계에서 이미 생성된 `ml_dataset_radius_all_YYMMDD` 테이블명을 위젯으로 받아 **읽기만** 합니다.
8. 출력 테이블도 위젯에 입력된 명시적 테이블명을 사용합니다. 테이블명은 자동 생성하지 않습니다.
9. 최종 Delta 출력 컬럼은 아래 10개로 고정합니다.

| 컬럼명 | 타입 | 설명 |
|---|---|---|
| `farmId` | string | 농장 고유 식별자 |
| `riskDate` | date | 위험도 예측 기준 날짜 |
| `riskScore` | double | 모델이 계산한 위험도 점수 |
| `riskLevel` | string | HIGH / MEDIUM / LOW |
| `modelVersion` | string | 예측에 사용한 모델 식별값 |
| `lastUpdated` | timestamp | 예측 생성 시각 |
| `riskFactors` | string | SHAP Top 3 위험 요인 JSON |
| `predictionBatchId` | string | 한 번의 예측 실행 전체를 구분하는 Batch ID. 같은 Batch의 모든 농장에 동일 값 |
| `riskRank` | bigint | 같은 날짜 내 농장 위험도 순위 |
| `isTop20Risk` | boolean | 확정된 동점 정책 기준 상위 20% 대상 여부 |

## 기본 입력/출력

- 기본 입력 테이블: `dt4_team1_databricks.gold.ml_dataset_radius_all_<당일 YYMMDD>`
- 기본 출력 테이블: `dt4_team1_databricks.gold.farm_risk_predictions_<당일 YYMMDD>`
- 기본 모델 URI: `models:/m-18c3a7b87dc94a5bbcba7ec1c4269753`

In [ ]:

# ============================================================
# 0. 패키지 설치
# ============================================================
# Databricks 클러스터 재시작 후 최초 1회 실행하세요.
# 이 셀을 실행하면 Python이 재시작되므로, 재시작 후 다음 셀부터 순서대로 다시 실행하세요.

%pip install lightgbm==4.6.0 shap==0.46.0

dbutils.library.restartPython()


In [ ]:
# ============================================================
# 1. Widget 및 안전장치 설정
# ============================================================

# 중요:
# - 이 Job 2 노트북은 ml_dataset_radius_all_YYMMDD 테이블을 만들지 않습니다.
# - 앞 단계에서 이미 생성된 Gold ML Dataset 테이블을 input_table 위젯으로 받아 읽기만 합니다.
# - output_table도 자동 생성 이름을 만들지 않고, 위젯에 명시된 테이블명을 그대로 사용합니다.
# - 아래 기본값은 예시/현재 실행용 기본값이며, 날짜가 바뀌면 위젯에서 직접 바꾸세요.

DEFAULT_INPUT_TABLE = f"dt4_team1_databricks.gold.ml_dataset_radius_all_20251216"
DEFAULT_OUTPUT_TABLE = f"dt4_team1_databricks.gold.farm_risk_predictions_20251216"

# ------------------------------------------------------------
# 기본 실행 파라미터
# ------------------------------------------------------------
dbutils.widgets.text(
    "input_table",
    DEFAULT_INPUT_TABLE,
)

dbutils.widgets.text(
    "output_table",
    DEFAULT_OUTPUT_TABLE,
)

dbutils.widgets.text(
    "model_uri",
    "models:/m-18c3a7b87dc94a5bbcba7ec1c4269753",
)

# 비워두면 입력 테이블의 최신 reference_date 사용
# 실제 저장 시에는 명시 날짜 사용을 권장한다.
dbutils.widgets.text(
    "prediction_date",
    "",
)

# 비워두면 자동 생성된다.
# 같은 Batch를 재시도할 때는 기존 predictionBatchId를 다시 넣는다.
# predictionBatchId는 한 번의 예측 실행 전체를 구분하는 값이며,
# 같은 Batch의 모든 농장 결과에 동일하게 들어간다.
dbutils.widgets.text(
    "prediction_batch_id",
    "",
)

# ------------------------------------------------------------
# 저장 관련 안전장치
# ------------------------------------------------------------
dbutils.widgets.dropdown(
    "write_output",
    "false",
    ["false", "true"],
)

dbutils.widgets.dropdown(
    "allow_create_output_table",
    "false",
    ["false", "true"],
)

dbutils.widgets.dropdown(
    "confirm_input_table_operational",
    "false",
    ["false", "true"],
)

dbutils.widgets.dropdown(
    "confirm_output_table_name",
    "false",
    ["false", "true"],
)

dbutils.widgets.dropdown(
    "confirm_top20_tie_policy",
    "false",
    ["false", "true"],
)

dbutils.widgets.dropdown(
    "confirm_local_env_for_write",
    "false",
    ["false", "true"],
)

# ------------------------------------------------------------
# 모델 실행 환경
# ------------------------------------------------------------
dbutils.widgets.dropdown(
    "model_env_manager",
    "local",
    ["local", "virtualenv"],
)

# ------------------------------------------------------------
# 위험 등급 및 상위 20% 기준
# riskLevel:
# - 상위 high_top_pct 이하: HIGH
# - 상위 medium_top_pct 이하: MEDIUM
# - 나머지: LOW
#
# isTop20Risk:
# - top20_risk_pct 기준
# - 동점 처리 정책은 top20_tie_policy로 결정
# ------------------------------------------------------------
dbutils.widgets.text(
    "high_top_pct",
    "0.20",
)

dbutils.widgets.text(
    "medium_top_pct",
    "0.50",
)

dbutils.widgets.text(
    "top20_risk_pct",
    "0.20",
)

dbutils.widgets.dropdown(
    "top20_tie_policy",
    "exact_count_farm_id",
    ["exact_count_farm_id", "include_all_ties"],
)

# ------------------------------------------------------------
# SHAP 설정
# ------------------------------------------------------------
dbutils.widgets.text(
    "shap_batch_size",
    "5000",
)

dbutils.widgets.dropdown(
    "risk_factor_selection_mode",
    "positive_then_abs",
    ["positive_then_abs", "abs_only"],
)

dbutils.widgets.dropdown(
    "risk_factor_weight_mode",
    "raw_shap",
    ["raw_shap", "abs_shap", "normalized_abs_shap"],
)

# ------------------------------------------------------------
# 선택적 예상 농장 수 검증
# 비워두면 검증하지 않음
# ------------------------------------------------------------
dbutils.widgets.text(
    "expected_farm_count",
    "",
)

# ============================================================
# Widget 값 읽기
# ============================================================
INPUT_TABLE = dbutils.widgets.get("input_table").strip()
OUTPUT_TABLE = dbutils.widgets.get("output_table").strip()
MODEL_URI = dbutils.widgets.get("model_uri").strip()
PREDICTION_DATE_TEXT = dbutils.widgets.get("prediction_date").strip()
PROVIDED_PREDICTION_BATCH_ID = dbutils.widgets.get("prediction_batch_id").strip()

WRITE_OUTPUT = dbutils.widgets.get("write_output").lower() == "true"
ALLOW_CREATE_OUTPUT_TABLE = dbutils.widgets.get("allow_create_output_table").lower() == "true"
CONFIRM_INPUT_TABLE_OPERATIONAL = dbutils.widgets.get("confirm_input_table_operational").lower() == "true"
CONFIRM_OUTPUT_TABLE_NAME = dbutils.widgets.get("confirm_output_table_name").lower() == "true"
CONFIRM_TOP20_TIE_POLICY = dbutils.widgets.get("confirm_top20_tie_policy").lower() == "true"
CONFIRM_LOCAL_ENV_FOR_WRITE = dbutils.widgets.get("confirm_local_env_for_write").lower() == "true"

MODEL_ENV_MANAGER = dbutils.widgets.get("model_env_manager")
HIGH_TOP_PCT = float(dbutils.widgets.get("high_top_pct"))
MEDIUM_TOP_PCT = float(dbutils.widgets.get("medium_top_pct"))
TOP20_RISK_PCT = float(dbutils.widgets.get("top20_risk_pct"))
TOP20_TIE_POLICY = dbutils.widgets.get("top20_tie_policy")
SHAP_BATCH_SIZE = int(dbutils.widgets.get("shap_batch_size"))
RISK_FACTOR_SELECTION_MODE = dbutils.widgets.get("risk_factor_selection_mode")
RISK_FACTOR_WEIGHT_MODE = dbutils.widgets.get("risk_factor_weight_mode")

expected_farm_count_text = dbutils.widgets.get("expected_farm_count").strip()
EXPECTED_FARM_COUNT = int(expected_farm_count_text) if expected_farm_count_text else None

if not INPUT_TABLE:
    raise ValueError("input_table이 비어 있습니다. 앞 단계에서 생성된 Gold ML Dataset 테이블명을 입력하세요.")

if not OUTPUT_TABLE:
    raise ValueError("output_table이 비어 있습니다. 저장할 Delta 테이블명을 입력하세요.")

if not MODEL_URI:
    raise ValueError("model_uri가 비어 있습니다.")

if not (0 < HIGH_TOP_PCT <= 1):
    raise ValueError("high_top_pct는 0보다 크고 1 이하이어야 합니다.")

if not (0 < MEDIUM_TOP_PCT <= 1):
    raise ValueError("medium_top_pct는 0보다 크고 1 이하이어야 합니다.")

if HIGH_TOP_PCT > MEDIUM_TOP_PCT:
    raise ValueError("high_top_pct는 medium_top_pct보다 작거나 같아야 합니다.")

if not (0 < TOP20_RISK_PCT <= 1):
    raise ValueError("top20_risk_pct는 0보다 크고 1 이하이어야 합니다.")

if TOP20_TIE_POLICY not in {"exact_count_farm_id", "include_all_ties"}:
    raise ValueError(f"지원하지 않는 top20_tie_policy입니다: {TOP20_TIE_POLICY}")

if SHAP_BATCH_SIZE <= 0:
    raise ValueError("shap_batch_size는 1 이상이어야 합니다.")

print("=" * 100)
print("Job 2 실행 파라미터")
print("Input Table:", INPUT_TABLE)
print("Output Table:", OUTPUT_TABLE)
print("Model URI:", MODEL_URI)
print("Prediction Date:", PREDICTION_DATE_TEXT or "(latest reference_date)")
print("Prediction Batch ID:", PROVIDED_PREDICTION_BATCH_ID or "(auto-generate once per run)")
print("Write Output:", WRITE_OUTPUT)
print("Allow Create Output Table:", ALLOW_CREATE_OUTPUT_TABLE)
print("High Top Pct:", HIGH_TOP_PCT)
print("Medium Top Pct:", MEDIUM_TOP_PCT)
print("Top20 Risk Pct:", TOP20_RISK_PCT)
print("Top20 Tie Policy:", TOP20_TIE_POLICY)
print("Risk Factor Selection Mode:", RISK_FACTOR_SELECTION_MODE)
print("Risk Factor Weight Mode:", RISK_FACTOR_WEIGHT_MODE)
print("=" * 100)


In [ ]:

# ============================================================
# 2. Import 및 고정 상수
# ============================================================

import hashlib
import importlib.metadata as importlib_metadata
import json
import math
import re
import uuid
from datetime import datetime, timedelta, timezone
from pathlib import Path

import mlflow
import mlflow.pyfunc
import mlflow.sklearn
import numpy as np
import pandas as pd
import shap

from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

try:
    from scipy import sparse
except Exception:
    sparse = None

KST = timezone(timedelta(hours=9))
UTC = timezone.utc

# ------------------------------------------------------------
# 기본 컬럼
# ------------------------------------------------------------
FARM_ID_COLUMN = "farm_id"
DATE_COLUMN = "reference_date"
TARGET_COLUMN = "label_infected"

POSITIVE_LABEL = 1
POSITIVE_CLASS_INDEX = 1

# ------------------------------------------------------------
# Job 1 모델 계보
# ------------------------------------------------------------
SOURCE_RUN_ID = "7cb240095657469ab8b0dcf03dd2b89a"
SOURCE_MODEL_URI = "models:/m-b1c759b72a9f490aa1f0b1a518c9f853"
SIGNED_RUN_ID = "c6467528333c4996839925726278cf6f"
SIGNED_MODEL_URI = "models:/m-18c3a7b87dc94a5bbcba7ec1c4269753"

MODEL_FAMILY = "lightgbm"
MODEL_STRUCTURE = "sklearn_pipeline"
FEATURE_SET_NAME = "all_7d_plus_30d"
SELECTION_OBJECTIVE = "danger_recall_top20"
SELECTED_TRIAL_NUMBER = 29
RAW_FEATURE_COUNT = 30
TRANSFORMED_FEATURE_COUNT = 33

# ------------------------------------------------------------
# modelVersion
# ------------------------------------------------------------
# Registry Version을 아직 쓰지 않는 경우, 고정 Model URI의 hash를 모델 식별값으로 사용합니다.
MODEL_RELEASE_KEY = hashlib.sha256(MODEL_URI.encode("utf-8")).hexdigest()[:24]
MODEL_VERSION = MODEL_RELEASE_KEY

CALCULATED_AT_UTC = datetime.now(UTC)

if PROVIDED_PREDICTION_BATCH_ID:
    if re.fullmatch(r"[A-Za-z0-9._:-]{8,128}", PROVIDED_PREDICTION_BATCH_ID) is None:
        raise ValueError("prediction_batch_id 형식이 올바르지 않습니다.")
    PREDICTION_BATCH_ID = PROVIDED_PREDICTION_BATCH_ID
else:
    PREDICTION_BATCH_ID = (
        "pred_"
        + CALCULATED_AT_UTC.strftime("%Y%m%dT%H%M%SZ")
        + "_"
        + uuid.uuid4().hex[:8]
    )

print("=" * 100)
print("모델 및 Batch 정보")
print("Prediction Batch ID:", PREDICTION_BATCH_ID)
print("Calculated At UTC:", CALCULATED_AT_UTC.isoformat())
print("Model URI:", MODEL_URI)
print("Model Version:", MODEL_VERSION)
print("Model Release Key:", MODEL_RELEASE_KEY)
print("=" * 100)


In [ ]:

# ============================================================
# 3. factorCode-label-icon 매핑
# ============================================================

FACTOR_METADATA = {
    "bird_obs_count_30d_10km": {
        "label": "최근 30일 반경 10km 내 철새 관측 증가",
        "icon": "bird.png",
    },
    "bird_obs_count_30d_1km": {
        "label": "최근 30일 반경 1km 내 철새 관측 증가",
        "icon": "bird.png",
    },
    "bird_obs_count_30d_3km": {
        "label": "최근 30일 반경 3km 내 철새 관측 증가",
        "icon": "bird.png",
    },
    "bird_obs_count_30d_5km": {
        "label": "최근 30일 반경 5km 내 철새 관측 증가",
        "icon": "bird.png",
    },
    "bird_obs_count_30d_7km": {
        "label": "최근 30일 반경 7km 내 철새 관측 증가",
        "icon": "bird.png",
    },
    "bird_obs_count_7d_10km": {
        "label": "최근 7일 반경 10km 내 철새 관측 증가",
        "icon": "bird.png",
    },
    "bird_obs_count_7d_1km": {
        "label": "최근 7일 반경 1km 내 철새 관측 증가",
        "icon": "bird.png",
    },
    "bird_obs_count_7d_3km": {
        "label": "최근 7일 반경 3km 내 철새 관측 증가",
        "icon": "bird.png",
    },
    "bird_obs_count_7d_5km": {
        "label": "최근 7일 반경 5km 내 철새 관측 증가",
        "icon": "bird.png",
    },
    "bird_obs_count_7d_7km": {
        "label": "최근 7일 반경 7km 내 철새 관측 증가",
        "icon": "bird.png",
    },
    "duck_obs_count_30d_10km": {
        "label": "최근 30일 반경 10km 내 오리류 관측 증가",
        "icon": "duck.png",
    },
    "duck_obs_count_30d_1km": {
        "label": "최근 30일 반경 1km 내 오리류 관측 증가",
        "icon": "duck.png",
    },
    "duck_obs_count_30d_3km": {
        "label": "최근 30일 반경 3km 내 오리류 관측 증가",
        "icon": "duck.png",
    },
    "duck_obs_count_30d_5km": {
        "label": "최근 30일 반경 5km 내 오리류 관측 증가",
        "icon": "duck.png",
    },
    "duck_obs_count_30d_7km": {
        "label": "최근 30일 반경 7km 내 오리류 관측 증가",
        "icon": "duck.png",
    },
    "farm_bird_nearest_dist_km": {
        "label": "농장과 가장 가까운 철새 관측지점 거리",
        "icon": "map_pin.png",
    },
    "flock_size": {
        "label": "농장 사육 규모",
        "icon": "chicken.png",
    },
    "goose_obs_count_30d_10km": {
        "label": "최근 30일 반경 10km 내 기러기류 관측 증가",
        "icon": "goose.png",
    },
    "goose_obs_count_30d_1km": {
        "label": "최근 30일 반경 1km 내 기러기류 관측 증가",
        "icon": "goose.png",
    },
    "goose_obs_count_30d_3km": {
        "label": "최근 30일 반경 3km 내 기러기류 관측 증가",
        "icon": "goose.png",
    },
    "goose_obs_count_30d_5km": {
        "label": "최근 30일 반경 5km 내 기러기류 관측 증가",
        "icon": "goose.png",
    },
    "goose_obs_count_30d_7km": {
        "label": "최근 30일 반경 7km 내 기러기류 관측 증가",
        "icon": "goose.png",
    },
    "humidity": {
        "label": "습도 증가",
        "icon": "humidity.png",
    },
    "infected_farm_count_3km": {
        "label": "반경 3km 내 감염농장 증가",
        "icon": "virus.png",
    },
    "min_temp_7d": {
        "label": "최근 7일 최저기온 영향",
        "icon": "thermometer.png",
    },
    "outbreak_count_5yr": {
        "label": "최근 5년 지역 발생 이력 증가",
        "icon": "history.png",
    },
    "poultry_species_-1": {
        "label": "축종 정보 미확인",
        "icon": "poultry.png",
    },
    "poultry_species_0": {
        "label": "닭 농장 여부",
        "icon": "chicken.png",
    },
    "poultry_species_1": {
        "label": "오리 농장 여부",
        "icon": "duck.png",
    },
    "poultry_species_2": {
        "label": "기타 가금류 농장 여부",
        "icon": "poultry.png",
    },
    "precipitation_7d": {
        "label": "최근 7일 강수량 증가",
        "icon": "rain.png",
    },
    "wind_speed_avg_7d": {
        "label": "최근 7일 평균 풍속 영향",
        "icon": "wind.png",
    },
    "within_migratory_bird_site_10km": {
        "label": "철새도래지 10km 이내 위치",
        "icon": "wetland.png",
    },
}

print("Factor metadata count:", len(FACTOR_METADATA))


In [ ]:

# ============================================================
# 4. Utility functions
# ============================================================

def normalize_package_name(package_name):
    return re.sub(r"[-_.]+", "-", package_name).lower()


def resolve_installed_package_version(package_name):
    normalized_package_name = normalize_package_name(package_name)
    candidate_distribution_names = [package_name]

    if normalized_package_name == "mlflow":
        candidate_distribution_names = ["mlflow", "mlflow-skinny"]

    for distribution_name in candidate_distribution_names:
        try:
            installed_version = importlib_metadata.version(distribution_name)
            return installed_version, distribution_name
        except importlib_metadata.PackageNotFoundError:
            continue

    if normalized_package_name == "mlflow":
        module_version = getattr(mlflow, "__version__", None)
        if module_version is not None:
            return str(module_version), "mlflow.__version__"

    return None, None


def normalize_transformed_feature_name(feature_name):
    # ColumnTransformer/OHE feature name을 서비스 factorCode 후보로 정규화한다.
    name = str(feature_name)
    name = name.replace("num__", "").replace("cat__", "")

    # OneHotEncoder가 poultry_species_1.0 형태로 만들 경우 poultry_species_1로 정리
    m = re.fullmatch(r"poultry_species_(-?\d+)(?:\.0)?", name)
    if m:
        return f"poultry_species_{m.group(1)}"

    return name


def get_transformed_feature_names(preprocessor, transformed_feature_count):
    try:
        raw_names = list(preprocessor.get_feature_names_out())
        names = [normalize_transformed_feature_name(name) for name in raw_names]
    except Exception:
        names = [f"feature_{i}" for i in range(transformed_feature_count)]

    if len(names) != transformed_feature_count:
        names = [f"feature_{i}" for i in range(transformed_feature_count)]

    return names


def as_dense_array(matrix):
    if hasattr(matrix, "toarray"):
        return matrix.toarray()
    return np.asarray(matrix)


def extract_positive_shap_values(shap_values, positive_class_index):
    # SHAP version / LightGBM binary 출력 형태 차이를 흡수한다.
    if isinstance(shap_values, list):
        return np.asarray(shap_values[positive_class_index], dtype="float64")

    shap_array = np.asarray(shap_values, dtype="float64")

    # 일부 버전에서 (n_rows, n_features, n_classes) 형태 가능
    if shap_array.ndim == 3:
        return shap_array[:, :, positive_class_index]

    return shap_array


def make_weight(shap_value, abs_sum):
    if RISK_FACTOR_WEIGHT_MODE == "raw_shap":
        return float(shap_value)
    if RISK_FACTOR_WEIGHT_MODE == "abs_shap":
        return float(abs(shap_value))
    if RISK_FACTOR_WEIGHT_MODE == "normalized_abs_shap":
        if abs_sum <= 0:
            return 0.0
        return float(abs(shap_value) / abs_sum)
    raise ValueError(f"지원하지 않는 risk_factor_weight_mode={RISK_FACTOR_WEIGHT_MODE}")


def build_top3_risk_factors_for_row(shap_row, feature_value_row, feature_names):
    candidates = []
    abs_sum = float(np.abs(shap_row).sum())

    for idx, raw_factor_code in enumerate(feature_names):
        factor_code = normalize_transformed_feature_name(raw_factor_code)
        shap_value = float(shap_row[idx])
        feature_value = float(feature_value_row[idx]) if np.isfinite(feature_value_row[idx]) else np.nan

        # poultry_species one-hot feature는 해당 농장의 실제 활성 값이 아니면 제외
        if factor_code.startswith("poultry_species_") and feature_value < 0.5:
            continue

        metadata = FACTOR_METADATA.get(
            factor_code,
            {
                "label": factor_code,
                "icon": "info.png",
            },
        )

        candidates.append(
            {
                "factorCode": factor_code,
                "label": metadata["label"],
                "icon": metadata["icon"],
                "weight": make_weight(shap_value, abs_sum),
                "_shapValue": shap_value,
                "_absShapValue": abs(shap_value),
            }
        )

    if not candidates:
        return []

    if RISK_FACTOR_SELECTION_MODE == "abs_only":
        selected = sorted(candidates, key=lambda x: x["_absShapValue"], reverse=True)[:3]

    elif RISK_FACTOR_SELECTION_MODE == "positive_then_abs":
        positive_candidates = [candidate for candidate in candidates if candidate["_shapValue"] > 0]
        selected = sorted(positive_candidates, key=lambda x: x["_shapValue"], reverse=True)[:3]

        if len(selected) < 3:
            selected_codes = {item["factorCode"] for item in selected}
            remaining_candidates = [
                candidate
                for candidate in candidates
                if candidate["factorCode"] not in selected_codes
            ]
            supplement = sorted(
                remaining_candidates,
                key=lambda x: x["_absShapValue"],
                reverse=True,
            )[: 3 - len(selected)]
            selected.extend(supplement)

    else:
        raise ValueError(f"지원하지 않는 risk_factor_selection_mode={RISK_FACTOR_SELECTION_MODE}")

    public_factors = []
    for item in selected[:3]:
        public_factors.append(
            {
                "factorCode": item["factorCode"],
                "label": item["label"],
                "icon": item["icon"],
                "weight": round(float(item["weight"]), 8),
            }
        )

    return public_factors


def quote_identifier(table_name):
    # catalog.schema.table 형태를 backtick으로 안전하게 감싼다.
    return ".".join([f"`{part}`" for part in table_name.split(".")])


In [ ]:

# ============================================================
# 5. 모델 의존성과 현재 환경 비교
# ============================================================

model_artifact_directory = Path(
    mlflow.artifacts.download_artifacts(artifact_uri=MODEL_URI)
)

requirements_path = model_artifact_directory / "requirements.txt"

if not requirements_path.exists():
    raise FileNotFoundError(
        "모델 Artifact에서 requirements.txt를 찾지 못했습니다. "
        f"모델 경로={model_artifact_directory}"
    )

requirements_lines = [
    line.strip()
    for line in requirements_path.read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.strip().startswith("#")
]

model_requirement_versions = {}

for requirement_line in requirements_lines:
    match = re.match(r"^([A-Za-z0-9_.-]+)==([^;\s]+)", requirement_line)
    if match is None:
        continue

    raw_package_name = match.group(1)
    required_version = match.group(2)
    model_requirement_versions[normalize_package_name(raw_package_name)] = {
        "package_name": raw_package_name,
        "required_version": required_version,
    }

dependency_records = []

for normalized_name, requirement_info in model_requirement_versions.items():
    package_name = requirement_info["package_name"]
    required_version = requirement_info["required_version"]

    current_version, detected_from = resolve_installed_package_version(package_name)

    if current_version is None:
        status = "missing"
    elif current_version == required_version:
        status = "match"
    else:
        status = "version_mismatch"

    dependency_records.append(
        {
            "package_name": package_name,
            "required_version": required_version,
            "current_version": current_version,
            "detected_from": detected_from,
            "status": status,
        }
    )

dependency_report_pdf = (
    pd.DataFrame(dependency_records)
    .sort_values(by=["status", "package_name"])
    .reset_index(drop=True)
)

MODEL_REQUIREMENT_MISMATCHES = dependency_report_pdf[
    dependency_report_pdf["status"] != "match"
].to_dict(orient="records")

LOCAL_ENV_EXACT_MATCH = len(MODEL_REQUIREMENT_MISMATCHES) == 0

critical_package_names = {
    "mlflow",
    "lightgbm",
    "scikit-learn",
    "pandas",
    "cloudpickle",
    "numpy",
    "scipy",
}

LOCAL_ENV_CRITICAL_MISMATCHES = [
    record
    for record in MODEL_REQUIREMENT_MISMATCHES
    if normalize_package_name(record["package_name"]) in critical_package_names
]

missing_critical_packages = [
    record
    for record in LOCAL_ENV_CRITICAL_MISMATCHES
    if record["status"] == "missing"
]

NON_MLFLOW_REQUIREMENT_MISMATCHES = [
    record
    for record in MODEL_REQUIREMENT_MISMATCHES
    if normalize_package_name(record["package_name"]) not in {"mlflow", "mlflow-skinny"}
]

LOCAL_ENV_WRITE_CANDIDATE = (
    len(NON_MLFLOW_REQUIREMENT_MISMATCHES) == 0
    and len(missing_critical_packages) == 0
)

if MODEL_ENV_MANAGER == "local" and missing_critical_packages:
    raise RuntimeError(
        "Local 실행에 필요한 핵심 패키지가 누락됐습니다. "
        f"누락={missing_critical_packages}"
    )

print("=" * 100)
print("모델 의존성 점검")
print("Model Artifact:", model_artifact_directory)
print("Requirements:", requirements_path)
print("Environment Manager:", MODEL_ENV_MANAGER)
print("전체 버전 정확히 일치:", LOCAL_ENV_EXACT_MATCH)
print("MLflow 외 불일치 수:", len(NON_MLFLOW_REQUIREMENT_MISMATCHES))
print("Local 저장 검토 가능:", LOCAL_ENV_WRITE_CANDIDATE)
print("=" * 100)

display(dependency_report_pdf)


In [ ]:

# ============================================================
# 6. 모델 Signature 및 Pipeline 구조 검증
# ============================================================

model_info = mlflow.models.get_model_info(MODEL_URI)

if model_info.signature is None:
    raise ValueError(f"선택한 모델에 Signature가 없습니다. MODEL_URI={MODEL_URI}")

model_flavors = set(model_info.flavors.keys())

if "sklearn" not in model_flavors:
    raise ValueError(f"선택한 모델에 sklearn flavor가 없습니다. 현재 flavor={sorted(model_flavors)}")

local_native_model = mlflow.sklearn.load_model(MODEL_URI)

if not hasattr(local_native_model, "named_steps"):
    raise TypeError("로드한 모델이 sklearn Pipeline이 아닙니다.")

required_steps = {"preprocessor", "model"}
actual_steps = set(local_native_model.named_steps.keys())

if not required_steps.issubset(actual_steps):
    raise ValueError(
        "Pipeline 단계가 예상과 다릅니다. "
        f"필수={required_steps}, 현재={sorted(actual_steps)}"
    )

pipeline_preprocessor = local_native_model.named_steps["preprocessor"]
pipeline_estimator = local_native_model.named_steps["model"]

signature_input_specs = list(getattr(model_info.signature.inputs, "inputs", []))

FEATURE_COLUMNS = [
    str(input_spec.name)
    for input_spec in signature_input_specs
    if getattr(input_spec, "name", None) is not None
]

EXPECTED_FEATURE_COLUMNS = [
    "poultry_species",
    "flock_size",
    "farm_bird_nearest_dist_km",
    "within_migratory_bird_site_10km",
    "infected_farm_count_3km",
    "outbreak_count_5yr",
    "humidity",
    "min_temp_7d",
    "precipitation_7d",
    "wind_speed_avg_7d",
    "bird_obs_count_7d_1km",
    "bird_obs_count_30d_1km",
    "duck_obs_count_30d_1km",
    "goose_obs_count_30d_1km",
    "bird_obs_count_7d_3km",
    "bird_obs_count_30d_3km",
    "duck_obs_count_30d_3km",
    "goose_obs_count_30d_3km",
    "bird_obs_count_7d_5km",
    "bird_obs_count_30d_5km",
    "duck_obs_count_30d_5km",
    "goose_obs_count_30d_5km",
    "bird_obs_count_7d_7km",
    "bird_obs_count_30d_7km",
    "duck_obs_count_30d_7km",
    "goose_obs_count_30d_7km",
    "bird_obs_count_7d_10km",
    "bird_obs_count_30d_10km",
    "duck_obs_count_30d_10km",
    "goose_obs_count_30d_10km",
]

CATEGORICAL_FEATURES = ["poultry_species"]
NUMERIC_FEATURES = [
    feature_name
    for feature_name in EXPECTED_FEATURE_COLUMNS
    if feature_name not in CATEGORICAL_FEATURES
]

if FEATURE_COLUMNS != EXPECTED_FEATURE_COLUMNS:
    raise ValueError(
        "모델 Signature의 Feature 목록 또는 순서가 Job 1 확정값과 다릅니다.\n"
        f"Signature={FEATURE_COLUMNS}\n"
        f"Expected={EXPECTED_FEATURE_COLUMNS}"
    )

if len(FEATURE_COLUMNS) != RAW_FEATURE_COUNT:
    raise ValueError(
        "모델 입력 Feature 수가 30개가 아닙니다. "
        f"현재={len(FEATURE_COLUMNS)}"
    )

output_signature_specs = list(getattr(model_info.signature.outputs, "inputs", []))

if len(output_signature_specs) != 1:
    raise ValueError("모델 Output Signature 수가 예상과 다릅니다.")

output_shape = tuple(getattr(output_signature_specs[0], "shape", ()))

if output_shape != (-1, 2):
    raise ValueError(f"모델 출력 Shape가 (-1, 2)가 아닙니다. 현재={output_shape}")

model_classes = getattr(local_native_model, "classes_", None)

if model_classes is None:
    model_classes = getattr(pipeline_estimator, "classes_", None)

if model_classes is None:
    raise ValueError("모델 classes_를 확인할 수 없습니다.")

model_classes = list(np.asarray(model_classes))

positive_class_indices = [
    class_index
    for class_index, class_value in enumerate(model_classes)
    if str(class_value) == str(POSITIVE_LABEL)
]

if len(positive_class_indices) != 1:
    raise ValueError(f"양성 클래스 {POSITIVE_LABEL}의 위치를 찾지 못했습니다. classes_={model_classes}")

positive_class_index = positive_class_indices[0]

if positive_class_index != POSITIVE_CLASS_INDEX:
    raise ValueError(
        "양성 클래스 Index가 확정값과 다릅니다. "
        f"현재={positive_class_index}, 확정값={POSITIVE_CLASS_INDEX}"
    )

feature_order_pdf = pd.DataFrame(
    {
        "feature_order": range(1, len(FEATURE_COLUMNS) + 1),
        "feature_name": FEATURE_COLUMNS,
        "feature_type": [
            "categorical" if feature_name in CATEGORICAL_FEATURES else "numeric"
            for feature_name in FEATURE_COLUMNS
        ],
    }
)

print("=" * 100)
print("모델 Signature 및 Pipeline 검증 완료")
print("Model Flavors:", sorted(model_flavors))
print("Pipeline Type:", type(local_native_model))
print("Preprocessor Type:", type(pipeline_preprocessor))
print("Estimator Type:", type(pipeline_estimator))
print("Feature 수:", len(FEATURE_COLUMNS))
print("Output Shape:", output_shape)
print("Model Classes:", model_classes)
print("Positive Class Index:", positive_class_index)
print("=" * 100)

display(feature_order_pdf)


In [ ]:

# ============================================================
# 7. 입력 테이블 Profile 및 기준일 결정
# ============================================================

if not spark.catalog.tableExists(INPUT_TABLE):
    raise ValueError(f"입력 테이블이 존재하지 않습니다: {INPUT_TABLE}")

input_table_sdf = spark.table(INPUT_TABLE)
input_table_columns = set(input_table_sdf.columns)

required_input_columns = [FARM_ID_COLUMN, DATE_COLUMN] + FEATURE_COLUMNS
missing_input_columns = sorted(set(required_input_columns) - input_table_columns)

if missing_input_columns:
    raise ValueError(f"입력 테이블에 필수 컬럼이 없습니다: {missing_input_columns}")

input_table_sdf = input_table_sdf.withColumn(
    "_normalized_reference_date",
    F.to_date(F.col(DATE_COLUMN)),
)

date_range_row = (
    input_table_sdf
    .agg(
        F.min("_normalized_reference_date").alias("min_reference_date"),
        F.max("_normalized_reference_date").alias("max_reference_date"),
        F.count(F.lit(1)).alias("total_rows"),
        F.countDistinct(FARM_ID_COLUMN).alias("total_distinct_farms"),
    )
    .first()
)

LATEST_AVAILABLE_DATE = date_range_row["max_reference_date"]

if PREDICTION_DATE_TEXT:
    try:
        resolved_prediction_date = datetime.strptime(PREDICTION_DATE_TEXT, "%Y-%m-%d").date()
    except ValueError as error:
        raise ValueError(
            "prediction_date는 YYYY-MM-DD 형식이어야 합니다. "
            f"현재 값={PREDICTION_DATE_TEXT}"
        ) from error
    PREDICTION_DATE_MODE = "explicit_date"
else:
    resolved_prediction_date = LATEST_AVAILABLE_DATE
    if resolved_prediction_date is None:
        raise ValueError("유효한 reference_date를 찾지 못했습니다.")
    PREDICTION_DATE_MODE = "latest_date_fallback"

recent_date_profile_sdf = (
    input_table_sdf
    .filter(F.col("_normalized_reference_date").isNotNull())
    .groupBy("_normalized_reference_date")
    .agg(
        F.count(F.lit(1)).alias("row_count"),
        F.countDistinct(FARM_ID_COLUMN).alias("distinct_farm_count"),
        F.sum(
            F.when(F.col(FARM_ID_COLUMN).isNull(), F.lit(1)).otherwise(F.lit(0))
        ).alias("null_farm_id_count"),
    )
    .withColumn("duplicate_excess_rows", F.col("row_count") - F.col("distinct_farm_count"))
    .orderBy(F.col("_normalized_reference_date").desc())
    .limit(30)
)

print("=" * 100)
print("입력 테이블 Profile")
print("Input Table:", INPUT_TABLE)
print("Min Reference Date:", date_range_row["min_reference_date"])
print("Max Reference Date:", date_range_row["max_reference_date"])
print("Total Rows:", date_range_row["total_rows"])
print("Total Distinct Farms:", date_range_row["total_distinct_farms"])
print("Resolved Prediction Date:", resolved_prediction_date)
print("Prediction Date Mode:", PREDICTION_DATE_MODE)
print("=" * 100)

display(recent_date_profile_sdf)



In [ ]:

# ============================================================
# 8. 기준일 입력 Feature 조회 및 타입/중복 검증
# ============================================================

batch_input_sdf = (
    input_table_sdf
    .filter(F.col("_normalized_reference_date") == F.lit(resolved_prediction_date))
    .select(
        F.col(FARM_ID_COLUMN),
        F.col("_normalized_reference_date").alias(DATE_COLUMN),
        *[F.col(feature_name) for feature_name in FEATURE_COLUMNS],
    )
    .cache()
)

BATCH_INPUT_ROW_COUNT = batch_input_sdf.count()

if BATCH_INPUT_ROW_COUNT == 0:
    raise ValueError(f"선택한 reference_date의 입력 행이 없습니다. 날짜={resolved_prediction_date}")

BATCH_DISTINCT_FARM_COUNT = batch_input_sdf.select(FARM_ID_COLUMN).distinct().count()

if EXPECTED_FARM_COUNT is not None and BATCH_INPUT_ROW_COUNT != EXPECTED_FARM_COUNT:
    print(
        "주의: 입력 행 수가 expected_farm_count와 다릅니다. "
        f"expected={EXPECTED_FARM_COUNT}, actual={BATCH_INPUT_ROW_COUNT}. "
        "Dry Run은 가능하지만 저장 안전장치에서 차단됩니다."
    )

null_or_blank_farm_id_count = (
    batch_input_sdf
    .filter(
        F.col(FARM_ID_COLUMN).isNull()
        | (F.trim(F.col(FARM_ID_COLUMN).cast("string")) == F.lit(""))
    )
    .count()
)

if null_or_blank_farm_id_count > 0:
    raise ValueError(f"farm_id가 NULL 또는 빈 문자열인 행이 있습니다. 행 수={null_or_blank_farm_id_count}")

duplicate_farm_sdf = (
    batch_input_sdf
    .groupBy(FARM_ID_COLUMN)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_farm_count = duplicate_farm_sdf.count()

if duplicate_farm_count > 0:
    display(duplicate_farm_sdf.orderBy(F.col("count").desc()).limit(50))
    raise ValueError(f"동일 reference_date에 farm_id 중복이 있습니다. 중복 farm_id 수={duplicate_farm_count}")

# ------------------------------------------------------------
# 수치형 Cast 실패 검사
# 원본 값은 NULL이 아닌데 double 변환 후 NULL이면 실패로 판단
# ------------------------------------------------------------
cast_failure_aggregations = [
    F.sum(
        F.when(
            F.col(feature_name).isNotNull() & F.col(feature_name).cast("double").isNull(),
            F.lit(1),
        ).otherwise(F.lit(0))
    ).alias(f"{feature_name}__cast_failure")
    for feature_name in NUMERIC_FEATURES
]

cast_failure_row = batch_input_sdf.agg(*cast_failure_aggregations).first().asDict()

cast_failure_pdf = pd.DataFrame(
    [
        {
            "feature_name": feature_name,
            "cast_failure_count": int(cast_failure_row.get(f"{feature_name}__cast_failure", 0) or 0),
        }
        for feature_name in NUMERIC_FEATURES
    ]
)

TOTAL_CAST_FAILURE_COUNT = int(cast_failure_pdf["cast_failure_count"].sum())

if TOTAL_CAST_FAILURE_COUNT > 0:
    display(cast_failure_pdf[cast_failure_pdf["cast_failure_count"] > 0])
    raise ValueError(f"수치형 Feature의 타입 변환 실패 총계={TOTAL_CAST_FAILURE_COUNT}")

prepared_batch_sdf = (
    batch_input_sdf
    .select(
        F.col(FARM_ID_COLUMN).cast("string").alias(FARM_ID_COLUMN),
        F.col(DATE_COLUMN).cast("date").alias(DATE_COLUMN),
        *[
            (
                F.col(feature_name).cast("string").alias(feature_name)
                if feature_name in CATEGORICAL_FEATURES
                else F.col(feature_name).cast("double").alias(feature_name)
            )
            for feature_name in FEATURE_COLUMNS
        ],
    )
    .cache()
)

PREPARED_ROW_COUNT = prepared_batch_sdf.count()

if PREPARED_ROW_COUNT != BATCH_INPUT_ROW_COUNT:
    raise ValueError(
        "타입 변환 전후 행 수가 다릅니다. "
        f"변환 전={BATCH_INPUT_ROW_COUNT}, 변환 후={PREPARED_ROW_COUNT}"
    )

print("=" * 100)
print("기준일 입력 조회 및 타입 검증 완료")
print("Prediction Date:", resolved_prediction_date)
print("Input Rows:", BATCH_INPUT_ROW_COUNT)
print("Distinct Farms:", BATCH_DISTINCT_FARM_COUNT)
print("farm_id NULL/blank:", null_or_blank_farm_id_count)
print("farm_id duplicate:", duplicate_farm_count)
print("Cast Failure:", TOTAL_CAST_FAILURE_COUNT)
print("=" * 100)


In [ ]:

# ============================================================
# 9. Feature 결측 및 이상값 점검
# ============================================================

quality_aggregations = []

for feature_name in FEATURE_COLUMNS:
    quality_aggregations.append(
        F.sum(
            F.when(F.col(feature_name).isNull(), F.lit(1)).otherwise(F.lit(0))
        ).alias(f"{feature_name}__null")
    )

for feature_name in NUMERIC_FEATURES:
    quality_aggregations.append(
        F.sum(
            F.when(F.isnan(F.col(feature_name)), F.lit(1)).otherwise(F.lit(0))
        ).alias(f"{feature_name}__nan")
    )
    quality_aggregations.append(
        F.sum(
            F.when(
                (F.col(feature_name) == F.lit(float("inf")))
                | (F.col(feature_name) == F.lit(float("-inf"))),
                F.lit(1),
            ).otherwise(F.lit(0))
        ).alias(f"{feature_name}__infinite")
    )

quality_row = prepared_batch_sdf.agg(*quality_aggregations).first().asDict()

quality_records = []

for feature_name in FEATURE_COLUMNS:
    null_count = int(quality_row.get(f"{feature_name}__null", 0) or 0)
    nan_count = int(quality_row.get(f"{feature_name}__nan", 0) or 0)
    infinite_count = int(quality_row.get(f"{feature_name}__infinite", 0) or 0)

    quality_records.append(
        {
            "feature_name": feature_name,
            "feature_type": "categorical" if feature_name in CATEGORICAL_FEATURES else "numeric",
            "null_count": null_count,
            "null_rate": null_count / BATCH_INPUT_ROW_COUNT,
            "nan_count": nan_count,
            "infinite_count": infinite_count,
        }
    )

quality_pdf = pd.DataFrame(quality_records)

categorical_null_records = quality_pdf[
    (quality_pdf["feature_type"] == "categorical")
    & (quality_pdf["null_count"] > 0)
]

if not categorical_null_records.empty:
    display(categorical_null_records)
    raise ValueError("범주형 Feature에 NULL이 있습니다. 범주형 결측 정책 확인 후 실행하세요.")

invalid_numeric_records = quality_pdf[
    (quality_pdf["nan_count"] > 0)
    | (quality_pdf["infinite_count"] > 0)
]

if not invalid_numeric_records.empty:
    display(invalid_numeric_records)
    raise ValueError("수치형 Feature에 NaN 또는 무한대가 있습니다.")

all_null_records = quality_pdf[quality_pdf["null_count"] >= BATCH_INPUT_ROW_COUNT]

if not all_null_records.empty:
    display(all_null_records)
    raise ValueError("기준일 전체 행에서 값이 모두 NULL인 Feature가 있습니다.")

print("=" * 100)
print("Feature 품질 검사 완료")
print("수치형 일반 NULL은 Pipeline 내부 Imputer 처리 대상이므로 기록만 하고 허용합니다.")
print("=" * 100)

display(
    quality_pdf.sort_values(
        by=["null_rate", "feature_name"],
        ascending=[False, True],
    )
)


In [ ]:

# ============================================================
# 10. Native·PyFunc·Spark UDF Smoke Test
# ============================================================

MODEL_EXECUTION_VALIDATION_PASSED = False

smoke_input_sdf = (
    prepared_batch_sdf
    .orderBy(F.col(FARM_ID_COLUMN).asc())
    .limit(5)
    .cache()
)

SMOKE_ROW_COUNT = smoke_input_sdf.count()

if SMOKE_ROW_COUNT == 0:
    raise ValueError("Smoke Test 입력이 비어 있습니다.")

smoke_input_pdf = smoke_input_sdf.select(*FEATURE_COLUMNS).toPandas()

for feature_name in NUMERIC_FEATURES:
    smoke_input_pdf[feature_name] = pd.to_numeric(smoke_input_pdf[feature_name], errors="coerce").astype("float64")

for feature_name in CATEGORICAL_FEATURES:
    smoke_input_pdf[feature_name] = (
        smoke_input_pdf[feature_name]
        .map(lambda value: str(value) if pd.notna(value) else np.nan)
        .astype("object")
    )

if list(smoke_input_pdf.columns) != FEATURE_COLUMNS:
    raise ValueError("Smoke Test Feature 순서가 모델 Signature와 다릅니다.")

native_probabilities = np.asarray(
    local_native_model.predict_proba(smoke_input_pdf),
    dtype="float64",
)

local_pyfunc_model = mlflow.pyfunc.load_model(MODEL_URI)

pyfunc_probabilities = np.asarray(
    local_pyfunc_model.predict(smoke_input_pdf),
    dtype="float64",
)

expected_smoke_shape = (SMOKE_ROW_COUNT, 2)

if native_probabilities.shape != expected_smoke_shape:
    raise ValueError(f"Native 예측 Shape가 예상과 다릅니다. 현재={native_probabilities.shape}")

if pyfunc_probabilities.shape != expected_smoke_shape:
    raise ValueError(f"PyFunc 예측 Shape가 예상과 다릅니다. 현재={pyfunc_probabilities.shape}")

NATIVE_PYFUNC_MAX_ABS_DIFF = float(np.max(np.abs(native_probabilities - pyfunc_probabilities)))

if not np.allclose(native_probabilities, pyfunc_probabilities, rtol=1e-10, atol=1e-12):
    raise ValueError(
        "Native predict_proba와 PyFunc predict 결과가 다릅니다. "
        f"최대 절대 차이={NATIVE_PYFUNC_MAX_ABS_DIFF}"
    )

prediction_udf = mlflow.pyfunc.spark_udf(
    spark=spark,
    model_uri=MODEL_URI,
    result_type=T.ArrayType(T.DoubleType()),
    env_manager=MODEL_ENV_MANAGER,
)

model_input_struct = F.struct(
    *[
        F.col(feature_name).alias(feature_name)
        for feature_name in FEATURE_COLUMNS
    ]
)

spark_smoke_rows = (
    smoke_input_sdf
    .withColumn("_class_probabilities", prediction_udf(model_input_struct))
    .orderBy(F.col(FARM_ID_COLUMN).asc())
    .select(FARM_ID_COLUMN, DATE_COLUMN, "_class_probabilities")
    .collect()
)

spark_probabilities = np.asarray(
    [list(row["_class_probabilities"]) for row in spark_smoke_rows],
    dtype="float64",
)

if spark_probabilities.shape != expected_smoke_shape:
    raise ValueError(f"Spark UDF 출력 Shape가 예상과 다릅니다. 현재={spark_probabilities.shape}")

if not np.isfinite(spark_probabilities).all():
    raise ValueError("Spark UDF 결과에 NaN 또는 무한대가 있습니다.")

if (spark_probabilities < 0).any() or (spark_probabilities > 1).any():
    raise ValueError("Spark UDF 결과가 0~1 범위를 벗어났습니다.")

if not np.allclose(spark_probabilities.sum(axis=1), 1.0, rtol=1e-8, atol=1e-10):
    raise ValueError("Spark UDF의 행별 클래스 점수 합이 1이 아닙니다.")

PYFUNC_SPARK_MAX_ABS_DIFF = float(np.max(np.abs(pyfunc_probabilities - spark_probabilities)))

if not np.allclose(pyfunc_probabilities, spark_probabilities, rtol=1e-10, atol=1e-12):
    raise ValueError(
        "PyFunc와 Spark UDF 예측 결과가 다릅니다. "
        f"최대 절대 차이={PYFUNC_SPARK_MAX_ABS_DIFF}"
    )

MODEL_EXECUTION_VALIDATION_PASSED = True

smoke_test_result_pdf = pd.DataFrame(
    {
        "farm_id": [row[FARM_ID_COLUMN] for row in spark_smoke_rows],
        "class_0_score": spark_probabilities[:, 0],
        "class_1_score": spark_probabilities[:, POSITIVE_CLASS_INDEX],
        "probability_sum": spark_probabilities.sum(axis=1),
    }
)

print("=" * 100)
print("모델 실행 Smoke Test 성공")
print("Environment Manager:", MODEL_ENV_MANAGER)
print("Smoke Test Rows:", SMOKE_ROW_COUNT)
print("Native ↔ PyFunc 최대 차이:", NATIVE_PYFUNC_MAX_ABS_DIFF)
print("PyFunc ↔ Spark UDF 최대 차이:", PYFUNC_SPARK_MAX_ABS_DIFF)
print("=" * 100)

display(smoke_test_result_pdf)


In [ ]:

# ============================================================
# 11. 전체 배치 예측
# ============================================================

if not MODEL_EXECUTION_VALIDATION_PASSED:
    raise RuntimeError("Smoke Test가 완료되지 않았으므로 전체 배치 예측을 실행할 수 없습니다.")

scored_base_sdf = (
    prepared_batch_sdf
    .withColumn("_class_probabilities", prediction_udf(model_input_struct))
    .withColumn("_class_0_score", F.col("_class_probabilities").getItem(0).cast("double"))
    .withColumn("risk_score", F.col("_class_probabilities").getItem(POSITIVE_CLASS_INDEX).cast("double"))
    .cache()
)

SCORED_ROW_COUNT = scored_base_sdf.count()

if SCORED_ROW_COUNT != BATCH_INPUT_ROW_COUNT:
    raise ValueError(
        "입력 행 수와 예측 행 수가 다릅니다. "
        f"입력={BATCH_INPUT_ROW_COUNT}, 예측={SCORED_ROW_COUNT}"
    )

invalid_prediction_sdf = scored_base_sdf.filter(
    F.col("_class_probabilities").isNull()
    | (F.size(F.col("_class_probabilities")) != F.lit(2))
    | F.col("risk_score").isNull()
    | F.isnan(F.col("risk_score"))
    | (F.col("risk_score") < F.lit(0.0))
    | (F.col("risk_score") > F.lit(1.0))
    | (F.abs((F.col("_class_0_score") + F.col("risk_score")) - F.lit(1.0)) > F.lit(1e-6))
)

INVALID_PREDICTION_COUNT = invalid_prediction_sdf.count()

if INVALID_PREDICTION_COUNT > 0:
    display(invalid_prediction_sdf.limit(50))
    raise ValueError(f"모델 예측 결과에 이상이 있습니다. 이상 행 수={INVALID_PREDICTION_COUNT}")

risk_summary_row = (
    scored_base_sdf
    .agg(
        F.min("risk_score").alias("risk_min"),
        F.max("risk_score").alias("risk_max"),
        F.avg("risk_score").alias("risk_avg"),
        F.countDistinct("risk_score").alias("distinct_risk_scores"),
    )
    .first()
)

print("=" * 100)
print("전체 배치 예측 완료")
print("Input Rows:", BATCH_INPUT_ROW_COUNT)
print("Scored Rows:", SCORED_ROW_COUNT)
print("Invalid Predictions:", INVALID_PREDICTION_COUNT)
print("Risk Min:", risk_summary_row["risk_min"])
print("Risk Max:", risk_summary_row["risk_max"])
print("Risk Avg:", risk_summary_row["risk_avg"])
print("Distinct Risk Scores:", risk_summary_row["distinct_risk_scores"])
print("=" * 100)


In [ ]:

# ============================================================
# 12. SHAP Top 3 riskFactors 생성
# ============================================================
# 주의:
# - predict_proba는 Spark UDF로 전체 처리했습니다.
# - riskFactors는 농장별 설명값이 필요하므로 Native Pipeline 내부 preprocessor를 사용해 SHAP을 계산합니다.
# - 운영 예측에서 preprocessor를 새로 fit하지 않습니다.
# ============================================================

shap_input_pdf = (
    prepared_batch_sdf
    .orderBy(F.col(FARM_ID_COLUMN).asc())
    .select(FARM_ID_COLUMN, DATE_COLUMN, *FEATURE_COLUMNS)
    .toPandas()
)

if shap_input_pdf.empty:
    raise ValueError("SHAP 입력 데이터가 비어 있습니다.")

shap_feature_input_pdf = shap_input_pdf[FEATURE_COLUMNS].copy()

for feature_name in NUMERIC_FEATURES:
    shap_feature_input_pdf[feature_name] = pd.to_numeric(
        shap_feature_input_pdf[feature_name],
        errors="coerce",
    ).astype("float64")

for feature_name in CATEGORICAL_FEATURES:
    shap_feature_input_pdf[feature_name] = (
        shap_feature_input_pdf[feature_name]
        .map(lambda value: str(value) if pd.notna(value) else np.nan)
        .astype("object")
    )

if list(shap_feature_input_pdf.columns) != FEATURE_COLUMNS:
    raise ValueError("SHAP 입력 Feature 순서가 모델 Signature와 다릅니다.")

# Pipeline 내부 preprocessor를 그대로 사용
transformed_preview = pipeline_preprocessor.transform(shap_feature_input_pdf.head(min(5, len(shap_feature_input_pdf))))
transformed_feature_count = int(transformed_preview.shape[1])

if transformed_feature_count != TRANSFORMED_FEATURE_COUNT:
    raise ValueError(
        "Preprocessor 변환 후 Feature 수가 확정값 33개와 다릅니다. "
        f"현재={transformed_feature_count}"
    )

transformed_feature_names = get_transformed_feature_names(
    pipeline_preprocessor,
    transformed_feature_count,
)

explainer = shap.TreeExplainer(pipeline_estimator)

risk_factor_json_list = []
local_shap_debug_records = []

row_count = len(shap_feature_input_pdf)

for start_index in range(0, row_count, SHAP_BATCH_SIZE):
    end_index = min(start_index + SHAP_BATCH_SIZE, row_count)
    batch_raw_pdf = shap_feature_input_pdf.iloc[start_index:end_index].copy()

    batch_transformed = pipeline_preprocessor.transform(batch_raw_pdf)
    batch_transformed_array = as_dense_array(batch_transformed)

    if batch_transformed_array.shape[1] != len(transformed_feature_names):
        raise ValueError(
            "변환된 Feature 수와 Feature 이름 수가 다릅니다. "
            f"array={batch_transformed_array.shape[1]}, names={len(transformed_feature_names)}"
        )

    batch_shap_values = explainer.shap_values(batch_transformed_array)
    batch_shap_positive = extract_positive_shap_values(batch_shap_values, POSITIVE_CLASS_INDEX)

    if batch_shap_positive.shape != batch_transformed_array.shape:
        raise ValueError(
            "SHAP 값 Shape가 변환 Feature Shape와 다릅니다. "
            f"shap={batch_shap_positive.shape}, transformed={batch_transformed_array.shape}"
        )

    for local_row_index in range(batch_shap_positive.shape[0]):
        factors = build_top3_risk_factors_for_row(
            shap_row=batch_shap_positive[local_row_index],
            feature_value_row=batch_transformed_array[local_row_index],
            feature_names=transformed_feature_names,
        )

        if len(factors) != 3:
            raise ValueError(
                "riskFactors가 정확히 3개가 아닙니다. "
                f"row={start_index + local_row_index}, factors={factors}"
            )

        risk_factor_json_list.append(
            json.dumps(factors, ensure_ascii=False)
        )

        if len(local_shap_debug_records) < 100:
            for factor in factors:
                local_shap_debug_records.append(
                    {
                        "row_index": start_index + local_row_index,
                        "factorCode": factor["factorCode"],
                        "label": factor["label"],
                        "icon": factor["icon"],
                        "weight": factor["weight"],
                    }
                )

if len(risk_factor_json_list) != row_count:
    raise ValueError("SHAP riskFactors 행 수가 입력 행 수와 다릅니다.")

risk_factors_pdf = shap_input_pdf[[FARM_ID_COLUMN, DATE_COLUMN]].copy()
risk_factors_pdf["riskFactors"] = risk_factor_json_list

risk_factors_sdf = (
    spark.createDataFrame(risk_factors_pdf)
    .select(
        F.col(FARM_ID_COLUMN).cast("string").alias(FARM_ID_COLUMN),
        F.col(DATE_COLUMN).cast("date").alias(DATE_COLUMN),
        F.col("riskFactors").cast("string").alias("riskFactors"),
    )
    .cache()
)

RISK_FACTORS_ROW_COUNT = risk_factors_sdf.count()

if RISK_FACTORS_ROW_COUNT != BATCH_INPUT_ROW_COUNT:
    raise ValueError(
        "riskFactors 행 수가 입력 행 수와 다릅니다. "
        f"riskFactors={RISK_FACTORS_ROW_COUNT}, input={BATCH_INPUT_ROW_COUNT}"
    )

risk_factor_duplicate_count = (
    risk_factors_sdf
    .groupBy(FARM_ID_COLUMN, DATE_COLUMN)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if risk_factor_duplicate_count > 0:
    raise ValueError(f"riskFactors에 farm_id + reference_date 중복이 있습니다. 중복 수={risk_factor_duplicate_count}")

print("=" * 100)
print("SHAP Top 3 riskFactors 생성 완료")
print("Rows:", RISK_FACTORS_ROW_COUNT)
print("Transformed Feature Count:", transformed_feature_count)
print("Risk Factor Selection Mode:", RISK_FACTOR_SELECTION_MODE)
print("Risk Factor Weight Mode:", RISK_FACTOR_WEIGHT_MODE)
print("=" * 100)

display(pd.DataFrame(local_shap_debug_records).head(30))


In [ ]:
# ============================================================
# 13. 위험 순위, riskLevel, isTop20Risk, 서비스 Output 구성
# ============================================================

scored_with_factors_sdf = (
    scored_base_sdf
    .join(
        risk_factors_sdf,
        on=[FARM_ID_COLUMN, DATE_COLUMN],
        how="inner",
    )
    .cache()
)

SCORED_WITH_FACTORS_ROW_COUNT = scored_with_factors_sdf.count()

if SCORED_WITH_FACTORS_ROW_COUNT != BATCH_INPUT_ROW_COUNT:
    raise ValueError(
        "예측 결과와 riskFactors join 후 행 수가 입력 행 수와 다릅니다. "
        f"joined={SCORED_WITH_FACTORS_ROW_COUNT}, input={BATCH_INPUT_ROW_COUNT}"
    )

date_partition_window = Window.partitionBy(DATE_COLUMN)

risk_order_window = (
    Window
    .partitionBy(DATE_COLUMN)
    .orderBy(
        F.col("risk_score").desc(),
        F.col(FARM_ID_COLUMN).asc(),
    )
)

risk_score_tie_window = Window.partitionBy(DATE_COLUMN, "risk_score")

ranked_sdf = (
    scored_with_factors_sdf
    .withColumn("risk_rank", F.row_number().over(risk_order_window))
    .withColumn("batch_farm_count", F.count(F.lit(1)).over(date_partition_window))
    .withColumn("rank_pct", F.col("risk_rank") / F.col("batch_farm_count"))
    .withColumn(
        "priority_cutoff_count",
        F.greatest(
            F.lit(1),
            F.ceil(F.col("batch_farm_count") * F.lit(TOP20_RISK_PCT)).cast("int"),
        ),
    )
    .withColumn(
        "riskLevel",
        F.when(F.col("rank_pct") <= F.lit(HIGH_TOP_PCT), F.lit("HIGH"))
        .when(F.col("rank_pct") <= F.lit(MEDIUM_TOP_PCT), F.lit("MEDIUM"))
        .otherwise(F.lit("LOW")),
    )
    .withColumn("risk_score_tie_count", F.count(F.lit(1)).over(risk_score_tie_window))
)

# ------------------------------------------------------------
# 상위 20% 동점 처리 정책 계산
# 1) exact_count_farm_id: riskScore desc + farm_id asc 기준 정확히 N개만 True
# 2) include_all_ties: cutoff score와 같은 점수의 농장을 모두 True
# ------------------------------------------------------------
ranked_sdf = (
    ranked_sdf
    .withColumn(
        "top20_cutoff_score",
        F.max(
            F.when(
                F.col("risk_rank") == F.col("priority_cutoff_count"),
                F.col("risk_score"),
            )
        ).over(date_partition_window),
    )
    .withColumn(
        "is_top20_exact_count",
        F.col("risk_rank") <= F.col("priority_cutoff_count"),
    )
    .withColumn(
        "is_top20_include_all_ties",
        F.col("risk_score") >= F.col("top20_cutoff_score"),
    )
)

if TOP20_TIE_POLICY == "exact_count_farm_id":
    ranked_sdf = ranked_sdf.withColumn("isTop20Risk", F.col("is_top20_exact_count"))
elif TOP20_TIE_POLICY == "include_all_ties":
    ranked_sdf = ranked_sdf.withColumn("isTop20Risk", F.col("is_top20_include_all_ties"))
else:
    raise ValueError(f"지원하지 않는 TOP20_TIE_POLICY입니다: {TOP20_TIE_POLICY}")

# 상세 검증용 output
final_output_sdf = (
    ranked_sdf
    .select(
        F.col(FARM_ID_COLUMN),
        F.col(DATE_COLUMN),
        F.col("risk_score"),
        F.col("risk_rank"),
        F.col("rank_pct"),
        F.col("riskLevel"),
        F.col("riskFactors"),
        F.col("isTop20Risk"),
        F.col("priority_cutoff_count"),
        F.col("top20_cutoff_score"),
        F.col("is_top20_exact_count"),
        F.col("is_top20_include_all_ties"),
        F.lit(TOP20_RISK_PCT).cast("double").alias("top20_risk_pct"),
        F.lit(TOP20_TIE_POLICY).cast("string").alias("top20_tie_policy"),
        F.lit(PREDICTION_BATCH_ID).cast("string").alias("prediction_batch_id"),
        F.lit(MODEL_URI).cast("string").alias("model_uri"),
        F.lit(MODEL_RELEASE_KEY).cast("string").alias("model_release_key"),
        F.lit(MODEL_VERSION).cast("string").alias("model_version"),
        F.lit(MODEL_FAMILY).cast("string").alias("model_family"),
        F.lit(FEATURE_SET_NAME).cast("string").alias("feature_set_name"),
        F.lit(SELECTION_OBJECTIVE).cast("string").alias("selection_objective"),
        F.lit(SELECTED_TRIAL_NUMBER).cast("int").alias("selected_trial_number"),
        F.lit(INPUT_TABLE).cast("string").alias("input_table"),
        F.lit(SOURCE_RUN_ID).cast("string").alias("source_run_id"),
        F.lit(SIGNED_RUN_ID).cast("string").alias("signed_run_id"),
        F.lit(CALCULATED_AT_UTC).cast("timestamp").alias("calculated_at"),
    )
    .cache()
)

# 서비스/프론트 계약용 Delta output
DELTA_OUTPUT_COLUMNS = [
    "farmId",
    "riskDate",
    "riskScore",
    "riskLevel",
    "modelVersion",
    "lastUpdated",
    "riskFactors",
    "predictionBatchId",
    "riskRank",
    "isTop20Risk",
]

delta_output_sdf = (
    final_output_sdf
    .select(
        F.col(FARM_ID_COLUMN).cast("string").alias("farmId"),
        F.col(DATE_COLUMN).cast("date").alias("riskDate"),
        F.col("risk_score").cast("double").alias("riskScore"),
        F.col("riskLevel").cast("string").alias("riskLevel"),
        F.col("model_version").cast("string").alias("modelVersion"),
        F.col("calculated_at").cast("timestamp").alias("lastUpdated"),
        F.col("riskFactors").cast("string").alias("riskFactors"),
        F.col("prediction_batch_id").cast("string").alias("predictionBatchId"),
        F.col("risk_rank").cast("long").alias("riskRank"),
        F.col("isTop20Risk").cast("boolean").alias("isTop20Risk"),
    )
    .select(*DELTA_OUTPUT_COLUMNS)
    .cache()
)

FINAL_OUTPUT_ROW_COUNT = final_output_sdf.count()
DELTA_OUTPUT_ROW_COUNT = delta_output_sdf.count()

if FINAL_OUTPUT_ROW_COUNT != BATCH_INPUT_ROW_COUNT:
    raise ValueError(
        "상세 출력 행 수가 입력 행 수와 다릅니다. "
        f"입력={BATCH_INPUT_ROW_COUNT}, 상세 출력={FINAL_OUTPUT_ROW_COUNT}"
    )

if DELTA_OUTPUT_ROW_COUNT != BATCH_INPUT_ROW_COUNT:
    raise ValueError(
        "Delta 출력 행 수가 입력 행 수와 다릅니다. "
        f"입력={BATCH_INPUT_ROW_COUNT}, Delta 출력={DELTA_OUTPUT_ROW_COUNT}"
    )

if delta_output_sdf.columns != DELTA_OUTPUT_COLUMNS:
    raise ValueError(
        "Delta 출력 컬럼 또는 순서가 합의된 구조와 다릅니다.\n"
        f"예상={DELTA_OUTPUT_COLUMNS}\n"
        f"실제={delta_output_sdf.columns}"
    )

top20_count = delta_output_sdf.filter(F.col("isTop20Risk")).count()

print("=" * 100)
print("서비스 Output 구성 완료")
print("Rows:", DELTA_OUTPUT_ROW_COUNT)
print("Columns:", delta_output_sdf.columns)
print("Prediction Batch ID:", PREDICTION_BATCH_ID)
print("Top20 Tie Policy:", TOP20_TIE_POLICY)
print("Top20 Count:", top20_count)
print("=" * 100)

display(
    delta_output_sdf
    .orderBy(F.col("riskRank").asc(), F.col("farmId").asc())
    .limit(100)
)

In [ ]:
# ============================================================
# 14. 최종 Output 검증
# ============================================================

required_non_null_columns = [
    "farmId",
    "riskDate",
    "riskScore",
    "riskLevel",
    "modelVersion",
    "lastUpdated",
    "riskFactors",
    "predictionBatchId",
    "riskRank",
    "isTop20Risk",
]

required_null_condition = None

for column_name in required_non_null_columns:
    current_condition = F.col(column_name).isNull()
    required_null_condition = current_condition if required_null_condition is None else required_null_condition | current_condition

required_null_count = delta_output_sdf.filter(required_null_condition).count()

if required_null_count > 0:
    raise ValueError(f"Delta 필수 컬럼에 NULL이 있습니다. 행 수={required_null_count}")

invalid_risk_score_count = (
    delta_output_sdf
    .filter(
        F.col("riskScore").isNull()
        | F.isnan(F.col("riskScore"))
        | (F.col("riskScore") < F.lit(0.0))
        | (F.col("riskScore") > F.lit(1.0))
    )
    .count()
)

if invalid_risk_score_count > 0:
    raise ValueError(f"유효하지 않은 riskScore가 있습니다. 행 수={invalid_risk_score_count}")

invalid_risk_level_count = (
    delta_output_sdf
    .filter(~F.col("riskLevel").isin("HIGH", "MEDIUM", "LOW"))
    .count()
)

if invalid_risk_level_count > 0:
    raise ValueError(f"riskLevel이 HIGH/MEDIUM/LOW가 아닌 행이 있습니다. 행 수={invalid_risk_level_count}")

invalid_risk_rank_count = (
    delta_output_sdf
    .filter(F.col("riskRank").isNull() | (F.col("riskRank") < F.lit(1)))
    .count()
)

if invalid_risk_rank_count > 0:
    raise ValueError(f"riskRank가 유효하지 않은 행이 있습니다. 행 수={invalid_risk_rank_count}")

risk_factor_schema = T.ArrayType(
    T.StructType(
        [
            T.StructField("factorCode", T.StringType(), False),
            T.StructField("label", T.StringType(), False),
            T.StructField("icon", T.StringType(), False),
            T.StructField("weight", T.DoubleType(), False),
        ]
    )
)

risk_factor_check_sdf = (
    delta_output_sdf
    .withColumn("_riskFactorsParsed", F.from_json(F.col("riskFactors"), risk_factor_schema))
    .withColumn("_riskFactorCount", F.size(F.col("_riskFactorsParsed")))
)

invalid_risk_factor_count = (
    risk_factor_check_sdf
    .filter(
        F.col("_riskFactorsParsed").isNull()
        | (F.col("_riskFactorCount") != F.lit(3))
    )
    .count()
)

if invalid_risk_factor_count > 0:
    display(
        risk_factor_check_sdf
        .filter(F.col("_riskFactorsParsed").isNull() | (F.col("_riskFactorCount") != F.lit(3)))
        .limit(50)
    )
    raise ValueError(f"riskFactors JSON이 유효하지 않거나 3개가 아닙니다. 행 수={invalid_risk_factor_count}")

# 같은 Batch 안에서는 farmId가 유일해야 한다.
source_duplicate_count = (
    delta_output_sdf
    .groupBy("predictionBatchId", "farmId")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if source_duplicate_count > 0:
    raise ValueError(
        "Delta Source에 (predictionBatchId, farmId) 중복이 있습니다. "
        f"중복 Key 수={source_duplicate_count}"
    )

risk_date_distinct_count = delta_output_sdf.select("riskDate").distinct().count()
model_version_distinct_count = delta_output_sdf.select("modelVersion").distinct().count()
last_updated_distinct_count = delta_output_sdf.select("lastUpdated").distinct().count()
prediction_batch_distinct_count = delta_output_sdf.select("predictionBatchId").distinct().count()

if risk_date_distinct_count != 1:
    raise ValueError(f"한 Batch에 riskDate가 여러 개 존재합니다. 고유값 수={risk_date_distinct_count}")

if model_version_distinct_count != 1:
    raise ValueError(f"한 Batch에 modelVersion이 여러 개 존재합니다. 고유값 수={model_version_distinct_count}")

if last_updated_distinct_count != 1:
    raise ValueError(f"한 Batch에 lastUpdated가 여러 개 존재합니다. 고유값 수={last_updated_distinct_count}")

if prediction_batch_distinct_count != 1:
    raise ValueError(f"한 Batch에 predictionBatchId가 여러 개 존재합니다. 고유값 수={prediction_batch_distinct_count}")

# isTop20Risk 수 검증
TOP20_SELECTED_COUNT = delta_output_sdf.filter(F.col("isTop20Risk")).count()
EXPECTED_TOP20_EXACT_COUNT = max(1, int(math.ceil(DELTA_OUTPUT_ROW_COUNT * TOP20_RISK_PCT)))

if TOP20_TIE_POLICY == "exact_count_farm_id" and TOP20_SELECTED_COUNT != EXPECTED_TOP20_EXACT_COUNT:
    raise ValueError(
        "exact_count_farm_id 정책인데 isTop20Risk=True 행 수가 예상과 다릅니다. "
        f"예상={EXPECTED_TOP20_EXACT_COUNT}, 실제={TOP20_SELECTED_COUNT}"
    )

level_summary_sdf = (
    delta_output_sdf
    .groupBy("riskLevel")
    .agg(F.count(F.lit(1)).alias("row_count"))
    .orderBy("riskLevel")
)

print("=" * 100)
print("최종 Output 검증 완료")
print("Rows:", DELTA_OUTPUT_ROW_COUNT)
print("Required NULL Count:", required_null_count)
print("Invalid riskScore Count:", invalid_risk_score_count)
print("Invalid riskRank Count:", invalid_risk_rank_count)
print("Invalid riskFactors Count:", invalid_risk_factor_count)
print("Duplicate Source Key Count:", source_duplicate_count)
print("Prediction Batch ID:", PREDICTION_BATCH_ID)
print("Top20 Tie Policy:", TOP20_TIE_POLICY)
print("Top20 Selected Count:", TOP20_SELECTED_COUNT)
print("=" * 100)

display(level_summary_sdf)

In [ ]:
print("INPUT_TABLE =", repr(INPUT_TABLE))
print("OUTPUT_TABLE =", repr(OUTPUT_TABLE))
print("PREDICTION_DATE_TEXT =", repr(PREDICTION_DATE_TEXT))
print("ALLOW_CREATE_OUTPUT_TABLE =", ALLOW_CREATE_OUTPUT_TABLE)
print("spark.catalog.tableExists(OUTPUT_TABLE) =", spark.catalog.tableExists(OUTPUT_TABLE))

spark.sql("""
SHOW TABLES IN dt4_team1_databricks.gold LIKE 'farm_risk_predictions*'
""").show(truncate=False)

In [ ]:

# ============================================================
# 15. Delta 저장 전 최종 안전장치
# ============================================================

WRITE_GATE_PASSED = False

OUTPUT_TABLE_EXISTS = bool(OUTPUT_TABLE) and spark.catalog.tableExists(OUTPUT_TABLE)

if not WRITE_OUTPUT:
    print("=" * 100)
    print("WRITE_OUTPUT=False")
    print("Dry Run 모드이므로 Delta 테이블을 생성하거나 수정하지 않습니다.")
    print("=" * 100)

else:
    blocking_reasons = []

    if not CONFIRM_INPUT_TABLE_OPERATIONAL:
        blocking_reasons.append("운영 입력 Gold 테이블이 확정되지 않았습니다.")

    if not CONFIRM_OUTPUT_TABLE_NAME:
        blocking_reasons.append("출력 Delta 테이블명이 확인되지 않았습니다.")

    if not CONFIRM_TOP20_TIE_POLICY:
        blocking_reasons.append("상위 20% 동점 처리 정책이 확정되지 않았습니다.")

    if PREDICTION_DATE_MODE != "explicit_date":
        blocking_reasons.append("실제 저장 시 prediction_date를 명시적으로 입력해야 합니다.")

    if EXPECTED_FARM_COUNT is not None and BATCH_INPUT_ROW_COUNT != EXPECTED_FARM_COUNT:
        blocking_reasons.append(
            "실제 입력 행 수가 예상 농장 수와 다릅니다. "
            f"예상={EXPECTED_FARM_COUNT}, 실제={BATCH_INPUT_ROW_COUNT}"
        )

    if not MODEL_EXECUTION_VALIDATION_PASSED:
        blocking_reasons.append("Native·PyFunc·Spark UDF Smoke Test가 통과되지 않았습니다.")

    if MODEL_ENV_MANAGER == "local":
        if not LOCAL_ENV_WRITE_CANDIDATE:
            blocking_reasons.append(
                "MLflow 외 모델 패키지 버전이 현재 Local 환경과 일치하지 않습니다. "
                f"불일치={NON_MLFLOW_REQUIREMENT_MISMATCHES}"
            )

        if not CONFIRM_LOCAL_ENV_FOR_WRITE:
            blocking_reasons.append("Local 환경으로 실제 Delta 저장을 진행한다는 명시적 확인이 없습니다.")

    if DELTA_OUTPUT_ROW_COUNT != BATCH_INPUT_ROW_COUNT:
        blocking_reasons.append("Delta 출력 행 수가 입력 행 수와 다릅니다.")

    if delta_output_sdf.columns != DELTA_OUTPUT_COLUMNS:
        blocking_reasons.append("Delta 출력 컬럼이 최종 계약과 다릅니다.")

    if not OUTPUT_TABLE_EXISTS and not ALLOW_CREATE_OUTPUT_TABLE:
        blocking_reasons.append("출력 테이블이 아직 없지만 최초 테이블 생성이 허용되지 않았습니다.")

    if OUTPUT_TABLE_EXISTS and ALLOW_CREATE_OUTPUT_TABLE:
        blocking_reasons.append("출력 테이블이 이미 존재합니다. allow_create_output_table을 false로 변경해야 합니다.")

    if blocking_reasons:
        blocking_message = "\n".join([f"- {reason}" for reason in blocking_reasons])
        raise RuntimeError(
            "Delta 저장이 안전장치에 의해 차단됐습니다.\n"
            f"{blocking_message}"
        )

    WRITE_GATE_PASSED = True

    print("=" * 100)
    print("Delta 저장 전 안전장치 통과")
    print("Input Table:", INPUT_TABLE)
    print("Output Table:", OUTPUT_TABLE)
    print("Output Table Exists:", OUTPUT_TABLE_EXISTS)
    print("Prediction Date:", resolved_prediction_date)
    print("Rows:", DELTA_OUTPUT_ROW_COUNT)
    print("Environment Manager:", MODEL_ENV_MANAGER)
    print("=" * 100)


In [ ]:
# ============================================================
# 16. Delta 테이블 최초 생성 또는 MERGE
# ============================================================

# 같은 Batch 재시도 시 같은 농장 row만 update하기 위한 key.
# Batch ID가 다르면 같은 농장·같은 날짜라도 별도 실행 결과로 남길 수 있다.
MERGE_KEY_COLUMNS = [
    "predictionBatchId",
    "farmId",
]

EXPECTED_DELTA_TYPE_MAP = {
    "farmId": "string",
    "riskDate": "date",
    "riskScore": "double",
    "riskLevel": "string",
    "modelVersion": "string",
    "lastUpdated": "timestamp",
    "riskFactors": "string",
    "predictionBatchId": "string",
    "riskRank": "bigint",
    "isTop20Risk": "boolean",
}

WRITE_OPERATION = None

if not WRITE_OUTPUT:
    print("WRITE_OUTPUT=False이므로 Delta CREATE·MERGE를 실행하지 않았습니다.")

else:
    if not WRITE_GATE_PASSED:
        raise RuntimeError("Delta 저장 안전장치가 통과되지 않았습니다.")

    source_duplicate_count = (
        delta_output_sdf
        .groupBy(*MERGE_KEY_COLUMNS)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    if source_duplicate_count > 0:
        raise ValueError(f"Delta MERGE Source에 중복 Key가 있습니다. 중복 Key 수={source_duplicate_count}")

    table_created_in_this_run = False

    if not spark.catalog.tableExists(OUTPUT_TABLE):
        if not ALLOW_CREATE_OUTPUT_TABLE:
            raise RuntimeError("출력 테이블 최초 생성이 허용되지 않았습니다.")

        spark.sql(
            f"""
            CREATE TABLE {quote_identifier(OUTPUT_TABLE)}
            (
                `farmId` STRING NOT NULL
                    COMMENT '농장 고유 식별자',

                `riskDate` DATE NOT NULL
                    COMMENT '위험도 예측 기준 날짜',

                `riskScore` DOUBLE NOT NULL
                    COMMENT '모델이 계산한 위험도 점수',

                `riskLevel` STRING NOT NULL
                    COMMENT '위험 등급. HIGH, MEDIUM, LOW',

                `modelVersion` STRING NOT NULL
                    COMMENT '예측에 사용한 모델의 고정 식별값',

                `lastUpdated` TIMESTAMP NOT NULL
                    COMMENT '예측 생성 또는 업데이트 시각',

                `riskFactors` STRING NOT NULL
                    COMMENT 'SHAP 기반 상위 3개 위험 요인 JSON 문자열',

                `predictionBatchId` STRING NOT NULL
                    COMMENT 'Job 2 한 번의 예측 실행 식별자. 같은 Batch의 모든 농장에 동일 값',

                `riskRank` BIGINT NOT NULL
                    COMMENT '동일 기준일 내 위험도 순위',

                `isTop20Risk` BOOLEAN NOT NULL
                    COMMENT '확정된 동점 처리 정책 기준 상위 20% 위험 대상 여부'
            )
            USING DELTA
            COMMENT '농장별 HPAI 위험도 예측 및 XAI 요인 결과'
            """
        )

        table_created_in_this_run = True

    target_sdf = spark.table(OUTPUT_TABLE)

    target_column_order = [field.name for field in target_sdf.schema.fields]
    target_type_map = {field.name: field.dataType.simpleString() for field in target_sdf.schema.fields}

    if target_column_order != DELTA_OUTPUT_COLUMNS:
        raise ValueError(
            "기존 Delta 테이블의 컬럼 또는 순서가 최종 출력 계약과 다릅니다.\n"
            f"예상={DELTA_OUTPUT_COLUMNS}\n"
            f"실제={target_column_order}"
        )

    if target_type_map != EXPECTED_DELTA_TYPE_MAP:
        raise ValueError(
            "기존 Delta 테이블의 데이터 타입이 최종 출력 계약과 다릅니다.\n"
            f"예상={EXPECTED_DELTA_TYPE_MAP}\n"
            f"실제={target_type_map}"
        )

    target_duplicate_count = (
        target_sdf
        .groupBy(*MERGE_KEY_COLUMNS)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )

    if target_duplicate_count > 0:
        raise ValueError("기존 Delta 테이블에 (predictionBatchId, farmId) 중복이 있습니다.")

    # 같은 predictionBatchId를 다른 날짜에 재사용하는 사고를 차단한다.
    existing_batch_dates = [
        row["riskDate"]
        for row in (
            target_sdf
            .filter(F.col("predictionBatchId") == F.lit(PREDICTION_BATCH_ID))
            .select("riskDate")
            .distinct()
            .collect()
        )
    ]

    if existing_batch_dates:
        if len(existing_batch_dates) != 1 or existing_batch_dates[0] != resolved_prediction_date:
            raise ValueError(
                "동일한 predictionBatchId가 다른 riskDate에 이미 사용됐습니다.\n"
                f"기존 날짜={existing_batch_dates}\n"
                f"현재 날짜={resolved_prediction_date}"
            )

    target_delta_table = DeltaTable.forName(spark, OUTPUT_TABLE)

    merge_condition = """
        target.predictionBatchId = source.predictionBatchId
        AND target.farmId = source.farmId
    """

    (
        target_delta_table
        .alias("target")
        .merge(
            delta_output_sdf.alias("source"),
            merge_condition,
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    WRITE_OPERATION = "CREATE_TABLE_AND_MERGE" if table_created_in_this_run else "MERGE_EXISTING_TABLE"

    print("=" * 100)
    print("Delta 저장 완료")
    print("Operation:", WRITE_OPERATION)
    print("Output Table:", OUTPUT_TABLE)
    print("Reference Date:", resolved_prediction_date)
    print("Prediction Batch ID:", PREDICTION_BATCH_ID)
    print("Model Version:", MODEL_VERSION)
    print("Written Source Rows:", DELTA_OUTPUT_ROW_COUNT)
    print("=" * 100)

In [ ]:
# ============================================================
# 17. 저장 결과 검증 및 Job 3 전달 값 설정
# ============================================================

if not WRITE_OUTPUT:
    print("WRITE_OUTPUT=False이므로 저장 검증과 Job Task Value 설정은 생략합니다.")

else:
    written_result_sdf = (
        spark.table(OUTPUT_TABLE)
        .filter(F.col("predictionBatchId") == F.lit(PREDICTION_BATCH_ID))
        .cache()
    )

    written_row_count = written_result_sdf.count()

    if written_row_count != DELTA_OUTPUT_ROW_COUNT:
        raise ValueError(
            "Delta 저장 후 현재 Batch 행 수가 예상과 다릅니다. "
            f"예상={DELTA_OUTPUT_ROW_COUNT}, 실제={written_row_count}"
        )

    written_duplicate_count = (
        written_result_sdf
        .groupBy("predictionBatchId", "farmId")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    if written_duplicate_count > 0:
        raise ValueError("저장 결과에 (predictionBatchId, farmId) 중복이 있습니다.")

    written_reference_dates = [
        row["riskDate"]
        for row in written_result_sdf.select("riskDate").distinct().collect()
    ]

    if written_reference_dates != [resolved_prediction_date]:
        raise ValueError(
            "저장 결과의 riskDate가 예상과 다릅니다.\n"
            f"예상={resolved_prediction_date}\n"
            f"실제={written_reference_dates}"
        )

    written_model_versions = [
        row["modelVersion"]
        for row in written_result_sdf.select("modelVersion").distinct().collect()
    ]

    if written_model_versions != [MODEL_VERSION]:
        raise ValueError(
            "저장 결과의 modelVersion이 예상과 다릅니다.\n"
            f"예상={MODEL_VERSION}\n"
            f"실제={written_model_versions}"
        )

    top20_written_count = written_result_sdf.filter(F.col("isTop20Risk")).count()

    task_values_set = False
    task_values_error = None

    try:
        dbutils.jobs.taskValues.set(key="prediction_batch_id", value=PREDICTION_BATCH_ID)
        dbutils.jobs.taskValues.set(key="prediction_date", value=str(resolved_prediction_date))
        dbutils.jobs.taskValues.set(key="output_table", value=OUTPUT_TABLE)
        dbutils.jobs.taskValues.set(key="model_version", value=MODEL_VERSION)
        dbutils.jobs.taskValues.set(key="prediction_row_count", value=int(written_row_count))
        dbutils.jobs.taskValues.set(key="top20_row_count", value=int(top20_written_count))
        dbutils.jobs.taskValues.set(key="last_updated", value=CALCULATED_AT_UTC.isoformat())
        task_values_set = True

    except Exception as error:
        task_values_error = str(error)
        print("대화형 Notebook 실행으로 판단되어 Job Task Value 설정을 건너뜁니다.")

    print("=" * 100)
    print("Job 2 저장 결과 검증 완료")
    print("Output Table:", OUTPUT_TABLE)
    print("Written Rows:", written_row_count)
    print("Duplicate Keys:", written_duplicate_count)
    print("Top20 Rows:", top20_written_count)
    print("Prediction Batch ID:", PREDICTION_BATCH_ID)
    print("Model Version:", MODEL_VERSION)
    print("Task Values Set:", task_values_set)
    print("Task Values Error:", task_values_error)
    print("=" * 100)

    display(
        written_result_sdf
        .orderBy(F.col("riskRank").asc(), F.col("farmId").asc())
        .limit(100)
    )

In [ ]:

# ============================================================
# 18. Dry Run용 Global Temp View 생성
# ============================================================

JOB3_DRYRUN_VIEW_NAME = "job2_predictions_service_output_dryrun"
JOB3_DRYRUN_VIEW = f"global_temp.{JOB3_DRYRUN_VIEW_NAME}"

if WRITE_OUTPUT:
    print("실제 Delta 저장 모드이므로 Dry Run용 Global Temp View 생성을 건너뜁니다.")

else:
    delta_output_sdf.createOrReplaceGlobalTempView(JOB3_DRYRUN_VIEW_NAME)
    spark.catalog.cacheTable(JOB3_DRYRUN_VIEW)

    JOB3_DRYRUN_ROW_COUNT = spark.table(JOB3_DRYRUN_VIEW).count()

    print("=" * 100)
    print("Dry Run용 Global Temp View 생성 완료")
    print("View:", JOB3_DRYRUN_VIEW)
    print("Rows:", JOB3_DRYRUN_ROW_COUNT)
    print("Columns:", spark.table(JOB3_DRYRUN_VIEW).columns)
    print("Prediction Batch ID:", PREDICTION_BATCH_ID)
    print("주의: 같은 Compute에서만 조회 가능하고 Compute 종료·재시작 시 사라집니다.")
    print("=" * 100)

    display(
        spark.table(JOB3_DRYRUN_VIEW)
        .orderBy(F.col("riskRank").asc(), F.col("farmId").asc())
        .limit(100)
    )


In [ ]:

# ============================================================
# 19. Cache 해제
# ============================================================

cache_object_names = [
    "written_result_sdf",
    "delta_output_sdf",
    "final_output_sdf",
    "scored_with_factors_sdf",
    "risk_factors_sdf",
    "scored_base_sdf",
    "smoke_input_sdf",
    "prepared_batch_sdf",
    "batch_input_sdf",
]

for cache_object_name in cache_object_names:
    cache_object = globals().get(cache_object_name)
    if cache_object is None:
        continue

    try:
        cache_object.unpersist()
    except Exception:
        pass

print("Job 2 Cache 해제 완료")
